In [ ]:
# ================= GenAI MENTOR STUDIO
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio==5.49.1",
                "google-genai<2", "gTTS", "pillow", "pymupdf"], check=True)

import io, json, time, tempfile, math, html
from getpass import getpass
import gradio as gr
from PIL import Image
from gtts import gTTS
from google import genai
from google.genai import types

# ---------------------------------------------------------------- API key
API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Enter your Gemini API key (hidden): ").strip()
if not API_KEY:
    raise ValueError("No API key entered. Run the cell again.")

client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(timeout=90000))

# ------------------------------------------- Auto-pick a working fast model
PREFERRED = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.1-flash-lite",
             "gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-2.0-flash"]

def pick_models():
    try:
        avail = []
        for m in client.models.list():
            name = m.name.replace("models/", "")
            acts = getattr(m, "supported_actions", None) or []
            if "generateContent" in acts or not acts:
                avail.append(name)
        ordered = [m for m in PREFERRED if m in avail]
        extra = [m for m in avail if "flash" in m and "image" not in m
                 and "live" not in m and "tts" not in m and m not in ordered]
        chosen = ordered + extra[:2]
        if chosen:
            return chosen
    except Exception as e:
        print("Could not list models, using defaults:", e)
    return PREFERRED[:3]

MODELS = pick_models()
print("Models in use (in order):", MODELS)

# --------------------------------------------------------- Core AI function
def make_config(model, system, temperature, json_mode, thinking):
    kw = {"max_output_tokens": 8192}
    if system:
        kw["system_instruction"] = system
    if json_mode:
        kw["response_mime_type"] = "application/json"
    if model.startswith("gemini-3"):
        if thinking:                      # Gemini 3 thinks by default -> slow
            kw["thinking_config"] = types.ThinkingConfig(thinking_level="low")
    else:
        kw["temperature"] = temperature
        if thinking and "pro" not in model:
            kw["thinking_config"] = types.ThinkingConfig(thinking_budget=0)
    return types.GenerateContentConfig(**kw)

def stream_ai(contents, system=None, temperature=0.5, json_mode=False):
    """Yield text chunks as they arrive; falls back to other models on error."""
    last = None
    for model in MODELS:
        for thinking in (True, False):
            got = False
            try:
                for chunk in client.models.generate_content_stream(
                        model=model, contents=contents,
                        config=make_config(model, system, temperature, json_mode, thinking)):
                    t = getattr(chunk, "text", None)
                    if t:
                        got = True
                        yield t
                if got:
                    return
                raise ValueError("Empty response")
            except Exception as e:
                if got:
                    raise
                last = e
                if thinking and "thinking" in str(e).lower():
                    continue
                break
    raise RuntimeError(f"All models failed. Last error: {last}")

def ask(prompt, system=None, temperature=0.5, json_mode=False):
    return "".join(stream_ai(prompt, system, temperature, json_mode)).strip()

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw
        raw = raw.rsplit("```", 1)[0]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        s = [i for i in (raw.find("["), raw.find("{")) if i != -1]
        e = [i for i in (raw.rfind("]"), raw.rfind("}")) if i != -1]
        return json.loads(raw[min(s): max(e) + 1])

# ------------------------------------------------------------ 1. Mentor chat
PERSONAS = {
    "Patient Professor": "You are a patient university professor. Explain clearly with examples.",
    "Friendly Study Buddy": "You are a casual, encouraging study buddy. Use simple words and analogies.",
    "Socratic Coach": "You are a Socratic coach. Guide with short questions; reveal answers gradually.",
    "Exam Strategist": "You are an exam coach. Give concise, exam-ready answers, mnemonics and common mistakes.",
}

def chat(message, history, persona):
    if not message or not message.strip():
        yield history, ""
        return
    history = history + [{"role": "user", "content": message},
                         {"role": "assistant", "content": ""}]
    contents = [types.Content(role="user" if m["role"] == "user" else "model",
                              parts=[types.Part.from_text(text=m["content"])])
                for m in history[:-1][-14:] if m["content"]]
    system = PERSONAS[persona] + " Keep answers well structured and not longer than needed."
    try:
        for piece in stream_ai(contents, system, 0.6):
            history[-1]["content"] += piece
            yield history, ""
    except Exception as e:
        history[-1]["content"] = f"⚠️ {e}"
        yield history, ""

# ------------------------------------------------------------ 2. Vision tutor
def vision(image, question):
    if image is None:
        yield "⬆️ Upload an image first."
        return
    buf = io.BytesIO()
    image.convert("RGB").save(buf, format="JPEG", quality=90)
    q = question.strip() or "Explain what this image shows and teach me the key ideas."
    contents = [types.Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg"),
                types.Part.from_text(text=q)]
    system = ("You are a visual tutor. Describe what you see, then explain or solve step by step. "
              "If it is a diagram, label the parts. If it is a problem, show the working.")
    out = ""
    try:
        for piece in stream_ai(contents, system, 0.4):
            out += piece
            yield out
    except Exception as e:
        yield f"⚠️ {e}"

# ------------------------------------------------------------ 3. Quiz arena
NQ = 5

def make_quiz(topic, difficulty):
    hidden = [gr.update(visible=False, choices=[], value=None) for _ in range(NQ)]
    if not topic.strip():
        return [None, "⚠️ Enter a topic."] + hidden
    prompt = f"""Create exactly {NQ} multiple-choice questions about: {topic}
Difficulty: {difficulty}. Return ONLY JSON in this form:
{{"questions":[{{"question":"...","options":["...","...","...","..."],
"answer":"exact text of the correct option","explanation":"one or two sentences"}}]}}
Each question has exactly 4 options and exactly one correct answer."""
    try:
        data = parse_json(ask(prompt, temperature=0.4, json_mode=True))
        if isinstance(data, list):
            data = {"questions": data}
        qs = [q for q in data["questions"] if q.get("answer") in q.get("options", [])][:NQ]
        if not qs:
            raise ValueError("No valid questions were returned. Try again.")
    except Exception as e:
        return [None, f"⚠️ {e}"] + hidden
    radios = []
    for i in range(NQ):
        if i < len(qs):
            radios.append(gr.update(visible=True, choices=qs[i]["options"], value=None,
                                    label=f"Q{i + 1}. {qs[i]['question']}"))
        else:
            radios.append(gr.update(visible=False, choices=[], value=None))
    return [qs, f"✅ {len(qs)} questions ready - answer them and press **Submit**."] + radios

def grade(qs, *answers):
    if not qs:
        return "Generate a quiz first."
    score, lines = 0, []
    for i, q in enumerate(qs):
        ok = answers[i] == q["answer"]
        score += ok
        mark = "✅" if ok else "❌"
        lines.append(f"**{mark} Q{i + 1}. {q['question']}**  \n"
                     f"Your answer: {answers[i] or '—'}  \nCorrect: **{q['answer']}**  \n"
                     f"_{q.get('explanation', '')}_")
    pct = round(100 * score / len(qs))
    return f"## Score: {score}/{len(qs)} ({pct}%)\n\n" + "\n\n".join(lines)

# ------------------------------------------------------------ 4. Explain & listen
def explain_listen(topic, style, level):
    if not topic.strip():
        return "⚠️ Enter a topic.", None
    prompt = (f"Explain '{topic}' to a {level} learner as a {style}. "
              "Plain spoken English, no markdown symbols, no bullet characters, "
              "about 200 words, with one real-world example and a one-line recap.")
    try:
        script = ask(prompt, temperature=0.6)
    except Exception as e:
        return f"⚠️ {e}", None
    audio_path = None
    try:
        audio_path = os.path.join(tempfile.gettempdir(), f"explain_{int(time.time())}.mp3")
        gTTS(text=script[:4500], lang="en").save(audio_path)
    except Exception as e:
        script += f"\n\n(Audio failed: {e})"
    return script, audio_path


# ------------------------------------------------------------ 5. Chat with PDF
_pdf_cache = {}

def load_pdf(path):
    import pymupdf
    key = (path, os.path.getmtime(path))
    if key not in _pdf_cache:
        doc = pymupdf.open(path)
        parts = [f"[PAGE {i + 1}]\n{p.get_text('text').strip()}"
                 for i, p in enumerate(doc) if p.get_text("text").strip()]
        _pdf_cache[key] = ("\n\n".join(parts), len(doc))
    return _pdf_cache[key]

def pdf_ask(file, question):
    if not file:
        yield "⬆️ Upload a PDF first."
        return
    path = file if isinstance(file, str) else getattr(file, "name", str(file))
    try:
        text, pages = load_pdf(path)
    except Exception as e:
        yield f"⚠️ Could not read the PDF: {e}"
        return
    if not text.strip():
        yield "⚠️ No selectable text found (scanned PDFs need OCR)."
        return
    text = text[:40000]
    q = question.strip() or ("Summarize this document with headings, key points, "
                             "definitions and likely exam questions.")
    system = ("Answer ONLY from the document. Cite pages like [PAGE 3] when supported. "
              "If the document does not contain the answer, say so clearly.")
    prompt = f"QUESTION:\n{q}\n\nDOCUMENT ({pages} pages):\n{text}"
    out = ""
    try:
        for piece in stream_ai(prompt, system, 0.2):
            out += piece
            yield out
    except Exception as e:
        yield f"⚠️ {e}"

# ------------------------------------------------------------ 6. Mind map maker
PALETTE = ["#ef4444", "#f59e0b", "#10b981", "#06b6d4", "#8b5cf6", "#ec4899", "#84cc16", "#f97316"]

def _node(x, y, label, fill, size=13, bold=False, opacity=1.0):
    limit = 30 if bold else 20
    label = label if len(label) <= limit else label[:limit - 2] + "…"
    w = max(70, int(len(label) * size * 0.56) + 22)
    h = size + 16
    x = min(max(x, w / 2 + 4), 1000 - w / 2 - 4)
    y = min(max(y, h / 2 + 4), 700 - h / 2 - 4)
    weight = "700" if bold else "500"
    return (f'<rect x="{x - w / 2:.0f}" y="{y - h / 2:.0f}" width="{w}" height="{h}" rx="{h / 2:.0f}" '
            f'fill="{fill}" fill-opacity="{opacity}"/><text x="{x:.0f}" y="{y + size * 0.35:.0f}" text-anchor="middle" '
            f'font-family="Arial, sans-serif" font-size="{size}" font-weight="{weight}" '
            f'fill="#ffffff">{html.escape(label)}</text>'), x, y

def build_mindmap_svg(data):
    center = str(data.get("center", "Topic"))
    branches = data.get("branches", [])[:8]
    n = max(len(branches), 1)
    cx, cy = 500, 350
    lines, nodes = [], []
    for i, b in enumerate(branches):
        color = PALETTE[i % len(PALETTE)]
        ang = 2 * math.pi * i / n - math.pi / 2
        bx, by = cx + 210 * math.cos(ang), cy + 135 * math.sin(ang)
        kids = [str(k) for k in b.get("children", [])][:4]
        m = len(kids)
        spread = min(0.45, max(0.2, (2 * math.pi / n) * 0.85 / max(m, 1)))
        for j, kid in enumerate(kids):
            ka = ang + (j - (m - 1) / 2) * spread
            sc = 0.86 if j % 2 else 1.0  # stagger neighbours so labels do not collide
            kx, ky = cx + 405 * sc * math.cos(ka), cy + 265 * sc * math.sin(ka)
            svg, kx, ky = _node(kx, ky, kid, color, 12, False, 0.85)
            lines.append(f'<line x1="{bx:.0f}" y1="{by:.0f}" x2="{kx:.0f}" y2="{ky:.0f}" '
                         f'stroke="{color}" stroke-width="2" opacity="0.7"/>')
            nodes.append(svg)
        lines.append(f'<line x1="{cx}" y1="{cy}" x2="{bx:.0f}" y2="{by:.0f}" '
                     f'stroke="{color}" stroke-width="4"/>')
        svg, _, _ = _node(bx, by, str(b.get("name", "Branch")), color, 14, True)
        nodes.append(svg)
    csvg, _, _ = _node(cx, cy, center, "#4f46e5", 18, True)
    return ('<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="700" viewBox="0 0 1000 700" '
            'style="width:100%;height:auto;background:#f8fafc;border-radius:16px">'
            + "".join(lines) + "".join(nodes) + csvg + "</svg>")

def make_mindmap(topic, detail):
    if not topic.strip():
        return "⚠️ Enter a topic.", None
    prompt = f"""Create a mind map for: {topic}
Detail level: {detail}. Return ONLY JSON:
{{"center":"short title","branches":[{{"name":"short label","children":["short label","short label","short label"]}}]}}
Use 5 to 7 branches, each with 3 or 4 children. Every label must be 1-4 words."""
    try:
        data = parse_json(ask(prompt, temperature=0.5, json_mode=True))
        if isinstance(data, list):
            data = data[0]
        svg = build_mindmap_svg(data)
    except Exception as e:
        return f"⚠️ {e}", None
    path = os.path.join(tempfile.gettempdir(), f"mindmap_{int(time.time())}.svg")
    with open(path, "w", encoding="utf-8") as f:
        f.write(svg)
    return svg, path

# ------------------------------------------------------------------- UI
CSS = """
.gradio-container {max-width: 1100px !important}
#hero {padding: 22px; border-radius: 18px; color: white;
       background: linear-gradient(120deg, #4f46e5, #0ea5e9);}
#hero h1 {margin: 0; font-size: 2rem}
"""

with gr.Blocks(title="GenAI Mentor Studio", css=CSS, theme=gr.themes.Soft()) as demo:
    gr.HTML("<div id='hero'><h1>🎓 GenAI Mentor Studio</h1>"
            "<p>Chat · Vision · Quiz · Listen · PDF Q&A · Mind maps — powered by Gemini</p></div>")

    with gr.Tab("💬 Mentor Chat"):
        persona = gr.Dropdown(list(PERSONAS), value="Patient Professor", label="Mentor personality")
        bot = gr.Chatbot(type="messages", height=430, label="Mentor")
        msg = gr.Textbox(placeholder="Ask anything... (press Enter)", label="Your message")
        with gr.Row():
            send = gr.Button("Send", variant="primary")
            clear = gr.Button("Clear")
        msg.submit(chat, [msg, bot, persona], [bot, msg])
        send.click(chat, [msg, bot, persona], [bot, msg])
        clear.click(lambda: ([], ""), None, [bot, msg])

    with gr.Tab("👁️ Vision Tutor"):
        with gr.Row():
            with gr.Column():
                img = gr.Image(type="pil", label="Upload a diagram, notes, or problem")
                vq = gr.Textbox(label="Question (optional)", placeholder="Solve this / explain this diagram...")
                vbtn = gr.Button("Analyze image", variant="primary")
            with gr.Column():
                vout = gr.Markdown("The explanation will stream here.")
        vbtn.click(vision, [img, vq], vout)

    with gr.Tab("🧠 Quiz Arena"):
        with gr.Row():
            qtopic = gr.Textbox(label="Topic", placeholder="e.g. Transformers in NLP", scale=3)
            qdiff = gr.Radio(["Easy", "Medium", "Hard"], value="Medium", label="Difficulty", scale=2)
        qgen = gr.Button("Generate quiz", variant="primary")
        qstate = gr.State(None)
        qstatus = gr.Markdown()
        radios = [gr.Radio(choices=[], visible=False, label="") for _ in range(NQ)]
        qsubmit = gr.Button("Submit answers")
        qresult = gr.Markdown()
        qgen.click(make_quiz, [qtopic, qdiff], [qstate, qstatus] + radios)
        qsubmit.click(grade, [qstate] + radios, qresult)

    with gr.Tab("🎧 Explain & Listen"):
        with gr.Row():
            etopic = gr.Textbox(label="Topic", placeholder="e.g. Backpropagation", scale=3)
            estyle = gr.Dropdown(["short podcast monologue", "friendly story", "exam revision briefing"],
                                 value="short podcast monologue", label="Style", scale=2)
            elevel = gr.Dropdown(["beginner", "intermediate", "advanced"], value="beginner",
                                 label="Level", scale=1)
        ebtn = gr.Button("Explain it", variant="primary")
        etext = gr.Textbox(label="Script", lines=9)
        eaudio = gr.Audio(label="Listen", type="filepath")
        ebtn.click(explain_listen, [etopic, estyle, elevel], [etext, eaudio])

    with gr.Tab("📄 Chat with PDF"):
        with gr.Row():
            with gr.Column(scale=2):
                pfile = gr.File(label="Upload a text-based PDF", file_types=[".pdf"], type="filepath")
                pq = gr.Textbox(label="Question (leave empty to summarize)",
                                placeholder="What is the main idea of chapter 2?", lines=2)
                pbtn = gr.Button("Ask the document", variant="primary")
            with gr.Column(scale=3):
                pout = gr.Markdown("Answers with page references will stream here.")
        pbtn.click(pdf_ask, [pfile, pq], pout)
        pq.submit(pdf_ask, [pfile, pq], pout)

    with gr.Tab("🗺️ Mind Map Maker"):
        with gr.Row():
            mtopic = gr.Textbox(label="Topic", placeholder="e.g. Machine Learning", scale=3)
            mdetail = gr.Radio(["Simple", "Detailed"], value="Detailed", label="Detail", scale=1)
        mbtn = gr.Button("Create mind map", variant="primary")
        mview = gr.HTML()
        mfile = gr.File(label="Download as SVG")
        mbtn.click(make_mindmap, [mtopic, mdetail], [mview, mfile])
        mtopic.submit(make_mindmap, [mtopic, mdetail], [mview, mfile])

    gr.Markdown("<small>AI can make mistakes - verify important answers.</small>")

demo.queue().launch(share=True, debug=False)

Enter your Gemini API key (hidden): ··········
Models in use (in order): ['gemini-3.8-flash', 'gemini-3.5-flash', 'gemini-3.1-flash-lite', 'gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-flash-latest', 'gemini-flash-lite-latest']
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://031731131fdc574fa0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
